In [5]:
import pandas as pd
import numpy as np
from typing import List, Tuple
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import statistics

/opt/conda/lib/python3.10/site-packages/dask/array/chunk_types.py:110: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  import cupy


In [6]:
BASE_PATH = './data'

In [7]:
targets = pd.read_csv(f'{BASE_PATH}/train.csv')
A = pd.read_csv(f'{BASE_PATH}/train/A.csv')
B = pd.read_csv(f'{BASE_PATH}/train/B.csv')

In [8]:
A_target = targets[targets['Test']=='A']
B_target = targets[targets['Test']=='B']

In [ ]:
A_train = pd.merge(A, A_target, on = 'Test_id', how = 'left')
B_train = pd.merge(B, B_target, on = 'Test_id', how = 'left')

In [10]:
A_train

,Test_id,Test_x,PrimaryKey,Age,TestDate,A1-1,A1-2,A1-3,A1-4,A2-1,...,A7-1,A8-1,A8-2,A9-1,A9-2,A9-3,A9-4,A9-5,Test_y,Label
0,0x744773A3B58E27F1811B47B53331F272AB5E569A9F72...,A,0x744773A3B58E27F1811B47B53331F272AB5E569A9F72...,20a,201811,"2,2,1,2,1,2,1,1,2,1,2,1,1,2,1,2,2,1","1,3,3,2,3,3,2,2,3,3,2,1,2,1,1,1,2,1","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","29,33,56,64,5,-51,44,-1,0,31,30,5,67,33,43,21,...","1,1,2,3,1,2,2,3,3,1,1,3,2,2,1,2,3,3",...,15,0,1,4,16,0,5,7,A,0
1,0xDA7DE599194F2B4F9624ACBCE96E66E2F8B357A4DD97...,A,0xDA7DE599194F2B4F9624ACBCE96E66E2F8B357A4DD97...,20a,201811,"2,2,1,2,2,1,1,1,2,2,1,2,1,1,2,1,2,1","3,2,2,1,1,3,1,1,2,3,2,1,3,1,2,3,3,2","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","76,1,27,25,41,34,-24,7,18,85,-18,-21,31,-7,18,...","2,3,3,1,2,2,3,2,3,1,2,3,1,3,2,1,1,1",...,11,9,0,1,3,0,0,4,A,0
2,0x7DF700D37380AD164ADF79B485C3A84A571186E3DB27...,A,0x7DF700D37380AD164ADF79B485C3A84A571186E3DB27...,20a,201802,"1,1,2,1,2,2,2,2,1,2,1,1,1,1,1,2,2,2","2,3,3,1,1,2,1,1,3,2,1,2,2,3,1,2,3,3","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-1,22,0,-37,-21,7,-34,-21,-79,-26,-80,-23,-63,...","1,1,3,2,3,2,1,1,1,1,3,2,2,2,2,3,3,3",...,16,2,2,2,5,0,4,4,A,0
3,0xAB057D1B8C3A8FB6E99F1B6E0F95086150951BCA8630...,A,0xAB057D1B8C3A8FB6E99F1B6E0F95086150951BCA8630...,20a,201805,"2,2,2,2,1,2,1,1,2,2,1,1,1,1,2,1,2,1","1,3,3,1,2,2,2,3,2,3,2,1,1,1,2,3,1,3","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-60,25,-25,-34,-40,-60,-46,-79,-77,-51,-80,-62...","1,2,2,3,1,1,2,2,3,1,3,2,1,3,3,3,2,1",...,15,0,0,0,0,2,0,2,A,0
4,0xAD4EC2C13080BC51C988805376AB404CCD2BA53CB108...,A,0xAD4EC2C13080BC51C988805376AB404CCD2BA53CB108...,20a,201806,"2,2,1,1,2,1,1,2,1,1,1,1,2,2,1,2,2,2","1,3,1,3,2,1,1,1,3,2,2,2,2,3,3,3,2,1","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0","-42,-77,-33,-3,-38,-58,-88,-4,-28,-58,-40,-29,...","1,3,3,2,2,2,3,1,3,1,1,2,1,3,1,2,3,2",...,8,0,2,9,16,1,21,4,A,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647236,0x5CC9861BC48C709E0B363F8568B5DBDC335876A36F0B...,A,0x5CC9861BC48C709E0B363F8568B5DBDC335876A36F0B...,70b,202205,"2,2,1,1,1,2,1,1,1,2,1,1,2,2,2,2,1,2","3,2,1,2,3,2,3,1,2,1,2,1,2,1,3,3,3,1","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-17,-151,-54,-6,124,-83,56,-7,10,-21,61,1,-43,...","1,2,1,3,1,3,1,3,1,2,2,2,3,3,3,2,2,1",...,7,4,3,11,10,0,13,11,A,0
647237,0x10F17BF0CABB791E9AC441B9F5E4C8A6126E295125AA...,A,0x10F17BF0CABB791E9AC441B9F5E4C8A6126E295125AA...,70b,202209,"2,1,2,1,2,1,2,2,2,2,1,1,1,1,1,1,2,2","1,1,3,2,1,2,2,2,3,2,3,1,2,1,3,3,1,3","0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0","-77,5,-213,-97,-72,-97,-89,-197,-111,-146,-344...","3,3,3,2,3,1,2,2,1,2,3,1,2,1,1,1,2,3",...,2,6,4,34,14,16,33,18,A,0
647238,0x97471D5B563350E9ED0637C7DD18C4EB72F7EE3A6AF6...,A,0x97471D5B563350E9ED0637C7DD18C4EB72F7EE3A6AF6...,70b,202203,"2,1,2,1,1,2,2,1,2,1,1,1,1,2,2,2,2,1","2,1,3,3,2,3,2,3,1,1,1,2,2,3,2,1,1,3","1,1,1,0,1,0,1,1,0,1,1,0,0,0,0,0,1,0","639,-481,725,-233,-342,-256,-322,-848,-294,-33...","1,1,2,1,3,1,3,1,2,3,3,2,2,3,1,3,2,2",...,1,8,3,27,20,13,25,18,A,0
647239,0xF43480648818B07D8FA3C12CC364CA35AA777417E17C...,A,0xF43480648818B07D8FA3C12CC364CA35AA777417E17C...,70b,202205,"1,1,2,2,1,2,2,1,1,2,2,1,2,1,1,2,1,2","1,3,3,2,1,3,1,2,2,3,1,3,2,2,3,1,1,2","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-28,-3,-42,-3,-24,8,-8,-18,-6,-77,8,-54,35,44,...","2,2,3,1,3,3,2,3,1,1,3,1,1,3,2,2,2,1",...,6,9,2,8,4,3,1,8,A,0


In [11]:
A_numeric_cols = ['A1-4','A2-4','A3-7','A4-5']
B_numeric_cols = ['B1-2','B2-2','B3-2','B4-2','B5-2']

In [12]:
A_int_cols = ['A8-1','A8-2','A9-1','A9-2','A9-3','A9-4','A9-5']
B_int_cols = ['B9-1','B9-2','B9-3','B9-4','B9-5','B10-1','B10-2','B10-3','B10-4','B10-5','B10-6']

In [13]:
for col in A_int_cols:
    A_train[col] = A_train[col].astype('int')

for col in B_int_cols:
    B_train[col] = B_train[col].astype('category')

In [14]:
def feature_add_rp_time(df, cols):
    for col in cols:
        df[f'{col}_mean'] = df[col].apply(
            lambda x: sum(map(float, x.split(','))) / len(x.split(',')) 
            if isinstance(x, str) else x
        )
    return df

In [15]:
A_train = feature_add_rp_time(A_train,A_numeric_cols)
B_train = feature_add_rp_time(B_train,B_numeric_cols)

In [16]:
A_str_cols = A_train.select_dtypes(include='object').columns.tolist()
B_str_cols = B_train.select_dtypes(include='object').columns.tolist()

for col in A_str_cols:
    A_train[col] = A_train[col].astype('category')

for col in B_str_cols:
    B_train[col] = B_train[col].astype('category')

print(A_train.dtypes)
print(B_train.dtypes)

Test_id       category
Test_x        category
PrimaryKey    category
Age           category
TestDate         int64
A1-1          category
A1-2          category
A1-3          category
A1-4          category
A2-1          category
A2-2          category
A2-3          category
A2-4          category
A3-1          category
A3-2          category
A3-3          category
A3-4          category
A3-5          category
A3-6          category
A3-7          category
A4-1          category
A4-2          category
A4-3          category
A4-4          category
A4-5          category
A5-1          category
A5-2          category
A5-3          category
A6-1             int64
A7-1             int64
A8-1             int64
A8-2             int64
A9-1             int64
A9-2             int64
A9-3             int64
A9-4             int64
A9-5             int64
Test_y        category
Label            int64
A1-4_mean      float64
A2-4_mean      float64
A3-7_mean      float64
A4-5_mean      float64
dtype: obje

In [17]:
drops = ['Test_id','Test_x','Test_y','Label']

In [33]:
A_train.isna().sum().sum()

19

In [35]:
A_train = A_train.dropna()

## Score 함수 구현

In [85]:
def A1_score(df):
    scores = []
    
    for i in range(len(df)):
        a12_str = str(df['A1-2'].iloc[i]) if pd.notna(df['A1-2'].iloc[i]) else '0'
        a12_list = [int(x) for x in a12_str.split(',')]
        base_score = statistics.mean(a12_list)
        
        
        a13_str = str(df['A1-3'].iloc[i]) if pd.notna(df['A1-3'].iloc[i]) else '0'
        a13_list = [int(x) for x in a13_str.split(',')]
        penalty = a13_list.count(1)
        
        
        a14_str = str(df['A1-4'].iloc[i]) if pd.notna(df['A1-4'].iloc[i]) else '0'
        a14_list = [abs(int(x)) for x in a14_str.split(',')]
        a14_scores = []
        for x in a14_list:
            if 0 <= x < 10:
                a14_scores.append(6)
            elif 10 <= x < 30:
                a14_scores.append(5)
            elif 30 <= x < 50:
                a14_scores.append(4)
            elif 50 <= x < 100:
                a14_scores.append(1)
            elif 100 <= x < 200:
                a14_scores.append(0)
            elif 200 <= x < 300:
                a14_scores.append(-1)
            elif x >= 300:
                a14_scores.append(-2)
        a24_mean = statistics.mean(a14_scores)
        if penalty >= 3:
            total_score = 0
        else:
            total_score = base_score + a24_mean - penalty
        scores.append(total_score)
    
    df['A1_score'] = scores
    return df

In [86]:
def A2_score(df):
    scores = []

    for i in range(len(df)):
        # -------------------
        # 1️⃣ A2-1, A2-2 합산
        # -------------------
        a21_str = str(df['A2-1'].iloc[i]) if pd.notna(df['A2-1'].iloc[i]) else '0'
        a22_str = str(df['A2-2'].iloc[i]) if pd.notna(df['A2-2'].iloc[i]) else '0'
        a21_list = [int(x) for x in a21_str.split(',')]
        a22_list = [int(x) for x in a22_str.split(',')]
        base_score = statistics.mean(a21_list) + statistics.mean(a22_list)

        # -------------------
        # 2️⃣ A2-3 감점 (1의 개수)
        # -------------------
        a23_str = str(df['A2-3'].iloc[i]) if pd.notna(df['A2-3'].iloc[i]) else '0'
        a23_list = [int(x) for x in a23_str.split(',')]
        penalty = a23_list.count(1)
        # -------------------
        # 3️⃣ A2-4 반응시간 점수화
        # -------------------
        a24_str = str(df['A2-4'].iloc[i]) if pd.notna(df['A2-4'].iloc[i]) else '0'
        a24_list = [abs(int(x)) for x in a24_str.split(',')]
        a24_scores = []
        for x in a24_list:
            if 0 <= x < 10:
                a24_scores.append(6)
            elif 10 <= x < 30:
                a24_scores.append(5)
            elif 30 <= x < 50:
                a24_scores.append(4)
            elif 50 <= x < 100:
                a24_scores.append(1)
            elif 100 <= x < 200:
                a24_scores.append(0)
            elif 200 <= x < 300:
                a24_scores.append(-1)
            elif x >= 300:
                a24_scores.append(-2)
        a24_mean = statistics.mean(a24_scores)

        # -------------------
        # 4️⃣ 최종 점수 계산
        # -------------------
        if penalty >= 3:
            total_score = 0
        else:
            total_score = base_score + a24_mean - penalty
        scores.append(total_score)

    df['A2_score'] = scores
    return df

In [87]:
def A3_score(df):
    scores = []
    
    for i in range(len(df)):
        a31_str = str(df['A3-1'].iloc[i]) if pd.notna(df['A3-1'].iloc[i]) else '0'
        a31_list = [int(x) for x in a31_str.split(',')]
        
        base_score = statistics.mean(a31_list)
        
        a35_str = str(df['A3-5'].iloc[i]) if pd.notna(df['A3-5'].iloc[i]) else '0'
        a35_list = [int(x) for x in a35_str.split(',')]
        a35_scores = []
        for x in a35_list:
            if x == 1:
                s = 1
            elif x == 2:
                s = -1
            elif x == 3:
                s = 2
            else:
                s = 0
            a35_scores.append(s)
        
        a35_mean = statistics.mean(a35_scores)
        
        a36_str = str(df['A3-6'].iloc[i]) if pd.notna(df['A3-6'].iloc[i]) else '0'
        a36_list = [int(x) for x in a36_str.split(',')]
        penalty = a36_list.count(1)
        
        a37_str = str(df['A3-7'].iloc[i]) if pd.notna(df['A3-7'].iloc[i]) else '0'
        a37_list = [int(x) for x in a37_str.split(',')]
        
        a37_scores = [2000 - x for x in a37_list]
        for d in range(len(a36_list)):
            if a36_list[d]==1:
                a37_scores[d] = 0
        a37_score = statistics.mean(a37_scores)
        if penalty >= 3:
            total_score = 0
        else:
            total_score = base_score + a35_mean + a37_score
        scores.append(total_score)
    df['A3_score'] = scores
    return df

In [88]:
def A4_score(df):
    scores = []
    for i in range(len(df)):
        a44_str = str(df['A4-4'].iloc[i]) if pd.notna(df['A4-4'].iloc[i]) else '0'
        a44_list = [int(x) for x in a44_str.split(',')]
        penalty = a44_list.count(1)
            
        a45_str = str(df['A4-5'].iloc[i]) if pd.notna(df['A4-5'].iloc[i]) else '0'
        a45_list = [abs(int(x)) for x in a45_str.split(',')]
        a45_scores = [3000 - x for x in a45_list]

        for d in range(len(a44_list)):
            if a44_list[d]==1:
                a45_scores[d]=0
        
        a45_score = statistics.mean(a45_scores)
                
        if penalty >= 11:
            total_score = 0
        else:
            total_score = a45_score
        scores.append(total_score)
    df['A4_score'] = scores
    return df

In [89]:
def convert_age(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    if val.endswith('a'):
        return int(val[:-1]) + 3
    elif val.endswith('b'):
        return int(val[:-1]) + 7
    else:
        return float(val)
    
A_train['Age'] = A_train['Age'].apply(convert_age)

/tmp/ipykernel_718244/1404005066.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A_train['Age'] = A_train['Age'].apply(convert_age)


In [90]:
A_train = A1_score(A_train)
A_train = A2_score(A_train)
A_train = A3_score(A_train)
A_train = A4_score(A_train)

/tmp/ipykernel_718244/2439715465.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['A1_score'] = scores
/tmp/ipykernel_718244/493729006.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['A2_score'] = scores
/tmp/ipykernel_718244/3777909478.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

In [91]:
A_X = A_train.drop(columns=drops)
A_Y = A_train['Label']
B_X = B_train.drop(columns=drops)
B_Y = B_train['Label']

In [92]:
xa_train, xa_val, ya_train, ya_val = train_test_split(A_X, A_Y, test_size=0.2, random_state=42)
xb_train, xb_val, yb_train, yb_val = train_test_split(B_X, B_Y, test_size=0.2, random_state=42)

xa_train, xa_test, ya_train, ya_test = train_test_split(xa_train, ya_train, test_size=0.2, random_state=42)
xb_train, xb_test, yb_train, yb_test = train_test_split(xb_train, yb_train, test_size=0.2, random_state=42)

In [93]:
A_cats = A_X.select_dtypes(include='category').columns.tolist()
B_cats = B_X.select_dtypes(include='category').columns.tolist()

In [94]:
A_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

B_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [95]:
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    binids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for i in range(n_bins):
        bin_true = y_true[binids == i]
        bin_prob = y_prob[binids == i]
        if len(bin_true) > 0:
            acc = bin_true.mean()
            conf = bin_prob.mean()
            ece += np.abs(acc - conf) * len(bin_true) / len(y_true)
    return ece


def leaderboard_metric(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    ece = expected_calibration_error(y_true, y_pred)
    
    score = 0.5 * (1 - auc) + 0.25 * brier + 0.25 * ece
    
    return 'leaderboard_score', score, False

In [96]:
A_model.fit(
    xa_train, ya_train,
    categorical_feature=A_cats,
    eval_set=[(xa_val, ya_val)],
    eval_metric=leaderboard_metric
)

B_model.fit(
    xb_train, yb_train,
    categorical_feature=B_cats,
    eval_set=[(xb_val, yb_val)],
    eval_metric=leaderboard_metric
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 9345, number of negative: 404886
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.070110 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 67948
[LightGBM] [Info] Number of data points in the train set: 414231, number of used features: 43
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.022560 -> initscore=-3.768764
[LightGBM] [Info] Start training from score -3.768764
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,5
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [97]:
import joblib

joblib.dump(A_model, "./model/A_model.pkl")
joblib.dump(B_model, "./model/B_model.pkl")

['./model/B_model.pkl']